⸻

HTTP & REST Essentials — Notebook Cheat Sheet

Table of Contents
1.	Mental Model: Client · Server · Resource · API
2.	HTTP Methods (Safe & Idempotent)
3.	Status Codes (Field Guide)
4.	URL Anatomy: path · query · body & media types
5.	Key Headers (request & response)
6.	Pagination: offset/limit vs cursor
7.	Filtering & Sorting
8.	Auth Basics: API Key, Bearer/JWT, OAuth2
9.	CORS (browser-only cross-origin rules)
10.	API Versioning
11.	Webhooks (inbound events, signatures)
12.	OpenAPI: single source of truth
13.	Postman Practice Playbook
14.	Mini Spec & Examples (Orders API)
15.	Readiness Checklist
16.	Glossary

⸻

1) Mental Model: Client · Server · Resource · API 
•	HTTP = transport protocol (the postal service).
•	Server = program listening on a port; can serve HTML pages, JSON APIs, files, etc.
•	Resource = domain entity (e.g., user, order, ticket).
•	API = documented contract on the server describing routes (URLs), methods, params, bodies, responses.

Client (browser / script / app)  --HTTP-->  API server (routes)
                                  <--HTTP--
           /users/42 (HTML) vs /api/users/42 (JSON)

We operate on resource data (JSON fields), not HTML tags.

Request - sended by client to a server
Response - sended by server as an answer to a client's request

⸻

2) HTTP Methods (Safe & Idempotent) 

Method	Safe	Idempotent	Typical Use
GET	✔︎	✔︎	Read a resource/collection
HEAD	✔︎	✔︎	Headers only (no body), like GET ping
POST	✘	✘	Create or trigger an action
PUT	✘	✔︎	Replace resource (full update)
PATCH	✘	(varies)	Partial update
DELETE	✘	✔︎	Delete resource
OPTIONS	✔︎	✔︎	What’s allowed here? (CORS preflight)

Idempotent = repeating the same request leads to the same resulting state (not necessarily identical byte-for-byte response).

⸻

3) Status Codes (Field Guide) 

2xx success
•	200 OK — success with body
•	201 Created — new resource created (often with Location header)
•	204 No Content — success, empty body (e.g., after DELETE)

3xx redirects & caching
•	301/308 — permanent redirect; 302/307 — temporary redirect
•	304 Not Modified — client cache still valid (see ETag)

4xx client errors
•	400 Bad Request — malformed/invalid input
•	401 Unauthorized — missing/invalid token
•	403 Forbidden — authenticated but not allowed
•	404 Not Found — resource doesn’t exist (or hidden)
•	409 Conflict — state conflict (duplicate, version)
•	422 Unprocessable Entity — semantically invalid (failed business validation)
•	429 Too Many Requests — rate limit exceeded (see Retry-After)

5xx server errors
•	500+ — problems on server side

⸻

4) URL Anatomy: path · query · body & media types 
•	Path (address of the resource): /api/users/42
•	Query (filters/options): ?status=active&limit=20&sort=-created_at
(query does not change the resource itself, only how it’s selected/returned)

Request body (mainly in POST/PUT/PATCH):
•	application/json — primary format for APIs
•	application/x-www-form-urlencoded — classic HTML forms (key=value)
•	multipart/form-data — form fields with files (uploads)

⸻

5) Key Headers (request & response) 

Request
	•	Authorization: Bearer <token> — auth
	•	Accept: application/json — desired representation
	•	User-Agent, Accept-Language — client metadata

Response
	•	Content-Type: application/json — representation type
	•	Cache-Control, ETag / If-None-Match — caching & conditional requests
	•	Link — pagination hints (e.g., rel="next")
	•	Rate limiting (vendor-specific): X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset, plus Retry-After on 429
	•	X-Request-Id / Correlation-Id — tracing id

⸻

6) Pagination: offset/limit vs cursor 
	•	Offset/Limit: ?offset=40&limit=20 (simple, can “drift” on large changing datasets)
	•	Page/Per-page: ?page=3&per_page=20 (same drift issues)
	•	Cursor (recommended at scale): server returns an opaque next_cursor; client requests the next chunk after that cursor.

Example:

GET /orders?limit=20
{ "items": [...20...], "next_cursor": "eyJpZCI6IDEyMzQ1fQ==" }

GET /orders?limit=20&cursor=eyJpZCI6IDEyMzQ1fQ==
→ next 20 after id=12345


⸻

7) Filtering & Sorting 
	•	Filters target resource fields (not HTML):
/orders?status=open&customer_id=77
	•	Sorting: /orders?sort=created_at or /orders?sort=-created_at (desc)
	•	Multi-sort: /orders?sort=created_at,-total

Field names are defined by the API contract; they are not globally standardized.

⸻

8) Auth Basics: API Key, Bearer/JWT, OAuth2 
	•	API Key — simplest; send via header or query (prefer header)
	•	Bearer/JWT — token in Authorization: Bearer <token>; JWT is a self-contained signed token (claims: sub, exp, scope)
	•	OAuth2 (how you obtain tokens)
	•	Authorization Code + PKCE — user login → app exchanges code for tokens
	•	Client Credentials — server-to-server tokens (no user)

⸻

9) CORS (browser-only cross-origin rules) 
	•	Browser security policy: JS at https://a.com can’t read https://b.com unless b.com allows it (response headers Access-Control-Allow-*).
	•	Preflight: browser sends OPTIONS to check allowed methods/headers.
	•	Not relevant for server-to-server calls (Postman, Python, cURL).

⸻

10) API Versioning 
	•	Path versioning: /api/v1/... (simple & visible)
	•	Header/media-type versioning: more advanced
	•	Don’t break old clients; deprecate with a migration window.

⸻

11) Webhooks (inbound events, signatures) 
	•	Provider calls your endpoint on an event (e.g., payment succeeded)
	•	Signature (HMAC) header to verify authenticity
	•	Respond fast (2xx), push heavy work to background queue
	•	Ensure idempotency (dedupe re-deliveries)

⸻

12) OpenAPI: single source of truth 
	•	Machine-readable API contract: paths, methods, params, schemas, errors, security
	•	Enables Swagger UI docs, client/server generation, Postman import, contract tests

Tiny snippet:

openapi: 3.0.3
info: { title: Orders API, version: "1.0.0" }
paths:
  /orders:
    get:
      parameters:
        - in: query
          name: status
          schema: { type: string, enum: [open, closed] }
        - in: query
          name: limit
          schema: { type: integer, minimum: 1, maximum: 100, default: 20 }
      responses:
        "200":
          description: OK
          content:
            application/json:
              schema: { $ref: "#/components/schemas/OrderList" }
components:
  schemas:
    Order:
      type: object
      properties:
        id: { type: integer }
        status: { type: string }
        customer_id: { type: integer }
        total: { type: number, format: float }
        created_at: { type: string, format: date-time }
    OrderList:
      type: object
      properties:
        items:
          type: array
          items: { $ref: "#/components/schemas/Order" }
        next_cursor: { type: string, nullable: true }


⸻

13) Postman Practice Playbook 
	•	Build collections with environments & variables ({{baseUrl}}, {{token}})
	•	Add pre-request scripts (timestamps, HMAC), tests (status/headers/schema)
	•	Practice chaining (create → read → delete)
	•	Import OpenAPI → auto-generate requests
	•	Run via Newman (CLI) and export HTML reports

⸻

14) Mini Spec & Examples (Orders API) 

Typical requests

# List open orders, newest first, first page
curl -H "Accept: application/json" \
     "https://api.example.com/v1/orders?status=open&sort=-created_at&limit=20"

# Create an order (token-based)
curl -X POST "https://api.example.com/v1/orders" \
     -H "Authorization: Bearer $TOKEN" \
     -H "Content-Type: application/json" \
     -d '{"id":12345,"status":"open","customer_id":77,"total":199.9,"created_at":"2025-10-01T12:34:56Z"}'
# → 201 Created + Location: /v1/orders/12345

Python requests

import requests

BASE = "https://api.example.com/v1"
headers = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}

# Read with pagination cursor
r = requests.get(f"{BASE}/orders", params={"limit": 20}, headers=headers)
data = r.json()
next_cursor = data.get("next_cursor")

# Create
payload = {"id": 12345, "status": "open", "customer_id": 77, "total": 199.9, "created_at": "2025-10-01T12:34:56Z"}
c = requests.post(f"{BASE}/orders", json=payload, headers=headers)
assert c.status_code == 201


⸻

15) Readiness Checklist 
	•	Explain PUT vs PATCH; 401 vs 403 vs 404; 409 vs 422
	•	Use cursor pagination and read Link header
	•	Handle 429 with Retry-After and backoff
	•	Send/receive JSON with proper Content-Type/Accept
	•	Import OpenAPI into Postman and add tests
	•	Run Postman collection via Newman and keep report
	•	Build a small FastAPI CRUD with correct status codes & OpenAPI

⸻

16) Glossary 
	•	Resource — domain entity (user/order); represented as JSON in REST APIs.
	•	Representation — the format returned (JSON vs HTML).
	•	Idempotent — repeating the same request yields the same state.
	•	Cursor — opaque bookmark for pagination (“continue after this item”).
	•	Bearer/JWT — token carried in Authorization header.
	•	CORS — browser rule for cross-origin requests.
	•	OpenAPI — machine-readable API contract.

⸻



# **Postman**

### Collections
    in Postman collection is a list of multipple requests. Typically all of them connected to the same API 

Initial(shared) value - is value of variable which will NOT be used and WILL be shared to other users who has access to your collection for example. Not share private information in this field 
Current(value) - is value which you modify and which is used for your requests. 
***Variable syntax {{}}*** - {{variable_name}}

**Query params** - made by using ? in url
**Path params** - made by using /:variable_name. We can follow the name of variable from API documentation, but we can also use just any other variable since Postman will automatically substitue whole variable with ":" to the value of this variable to find a path and go to that destination
    ex: books/:bookId -> books/2



## Automate testing with API

    made to automate and time-save process of testing API changes and know the result. In perfect case scenario should take all manual work from us and let postman test it automatically

    running through the scripts and written in JS syntax.
    Custom tests mostly made to ensure and double check returning values and cover the potential mistakes that could be done 

## Collection Runner

    

## Excel useful functions

- COUNTIFS/SUMIFS(sum_range/count_range, criteria_range1,criteria1, criteria_range2, criteria2,...) - sum or count elements in specified range, only that values which pass criterias of ranges they are part of

- TRIM - clean all unneccessary whitespaces, analog of .strip() method in Python. 
- CLEAN - clean all characters from ASCII 0-31. Not removing spaces so in most cases is combined with with TRIM
  Hack! - TRIM + CLEAN + SUBSTITUTE -> TRIM(SUBSTITUTE(CLEAN(A1), CHAR(160), ""))

- XLOOKUP(lookup_value, lookup_range, result_range) - where lookup_range and result_range have to be same format(column or either row)

- INDEX + MATCH -> INDEX(searching_range, row_value, column_value) where:
    - row_value and column_value = MATCH(lookup_value, lookup_range, match_type)
        match_type is:
        1. 0 if exact match
        2. 1 smaller or equal to a lookup_value(standard match_type)
        3. -1 greater or equal to a lookup_value


